# RA2CE Case East Africa

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
file_path = Path(r"C:\python\moonshot2_east_africa\ea_coastal_floods\static\network\coastal_polygon.geojson")
if file_path.exists():
    coastal_polygon = gpd.read_file(file_path, driver = "GeoJSON")

coastal_polygon.set_crs('EPSG:4326')

coastal_polygon.explore()

In [ ]:
from ra2ce.ra2ce_handler import Ra2ceHandler # import the ra2cehandler to run ra2ce analyses


In [ ]:
# specify the name of the path to the project folder where you created the RA2CE folder setup
root_dir = Path(r"C:\python\moonshot2_east_africa\ea_coastal_floods")
assert root_dir.exists()

In [ ]:
_network_ini_name = "network.ini" # set the name for the network.ini settings file
network_ini = root_dir / _network_ini_name #let RA2CE find the network.ini
_analysis_ini_name = 'analysis.ini'
analysis_ini = root_dir / _analysis_ini_name
assert network_ini.is_file() # check whether the network.ini file exists

In [ ]:
handler = Ra2ceHandler(network=network_ini, analysis=analysis_ini)
handler.configure()
handler.run_analysis()

In [ ]:
base_graph_edges = Path(r"C:\python\moonshot2_east_africa\ea_coastal_floods\static\output_graph\base_graph_hazard_edges.gpkg")
edges_gdf = gpd.read_file(base_graph_edges, driver = "GPKG")
edges_gdf.head()
print(type(edges_gdf))

In [ ]:
edges = edges_gdf.copy()

In [ ]:
column_names = ['EV1_ma', 'EV2_ma', 'EV3_ma', 'EV4_ma', 'EV5_ma', 'EV6_ma', 'EV7_ma', 'EV8_ma', 'EV9_ma', 'EV10_ma', 'EV11_ma']

def add_sum_column(column_names, geodataframe):
    geodataframe['sum_column'] = geodataframe[column_names].sum(axis=1)
    return geodataframe

# Then you can use the add_sum_column function
gdf = add_sum_column(column_names, edges)
gdf.head()




In [ ]:
import folium


def mute_basemap(map: folium.Map) -> folium.Map:
    map.get_root().header.add_child(
        folium.Element(
            "<style>.leaflet-tile-pane { filter: grayscale(1); opacity: 0.6; }</style>"
        ),
        name="muted_basemap",
    )
    return map


gdf_above0 = gdf[gdf['sum_column']>0]
import folium 

m = mute_basemap(edges.explore(color='darkgrey', tiles="Esri.WorldStreetMap"))
mute_basemap(gdf_above0.explore(m=m, column='sum_column', cmap='plasma_r', tiles="Esri.WorldStreetMap"))

folium.LayerControl().add_to(m)

m

In [ ]:
mute_basemap(gdf.explore(column='sum_column', cmap='gist_heat_r', tiles="Esri.WorldStreetMap"))